# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/-Explaining-Search-Performance-Gaps-Using-Ranking-Signals/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)
<!--  -->
This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import pandas as pd
from pathlib import Path

# Locate and load the dataset (supports both processed feature vector and raw parquet)
processed_path = Path("../data/processed/refresh_feature_vector.csv")
raw_path = Path("../data/raw/content_refresh_3m.parquet")
df = pd.read_csv(processed_path)
source_type = "Processed Feature Vector"

## 1. Unit of analysis + time window

- **One row = One unique content page** (uniquely identified by `content_hash_id`) associated with a specific website or account (`client_hash_id`).
- **Time Window:** The dataset spans a **3-month observation window** containing aggregated search performance metrics, content characteristics, and analytics logs.
- **Grain & Purpose:** Each row serves as an independent evaluation unit to assess whether a specific piece of content is underperforming relative to its peer group in search click-through rate (CTR).

In [10]:
import pandas as pd
from pathlib import Path



# Verify the grain (checking if content_hash_id is uniquely defining the row)
total_rows = len(df)
unique_contents = df["content_hash_id"].nunique()

print(f" Unique content_hash_id count: {unique_contents:,}")

if total_rows == unique_contents:
    print(" Grain Check Passed: Exactly one row per unique content item.")
else:
    print(" Grain Check Note: Duplicates found on content_hash_id. Checking client-content pairs...")
    unique_pairs = df[["client_hash_id", "content_hash_id"]].drop_duplicates().shape[0]
    print(f"   Unique (client_hash_id, content_hash_id) pairs: {unique_pairs:,}")

# Quick view of basic attributes to confirm scope
print("\n Sample identifiers and basic columns:")
display(df.head(3))

 Unique content_hash_id count: 118,092
 Grain Check Passed: Exactly one row per unique content item.

 Sample identifiers and basic columns:


,client_hash_id,content_hash_id,avg_position,total_clicks,total_impressions,ctr,days_with_data,ga4_sessions,engaged_sessions,scroll_events,...,days_since_update_vs_bucket_mean,search_volume_vs_bucket_median,search_volume_vs_bucket_mean,cpc_vs_bucket_median,cpc_vs_bucket_mean,freshness_x_demand,depth_vs_competition,age_x_optimization,maturity_x_freshness,opportunity_x_value
0,client_e547b89c05043229,content_d219af893b3d1d9b,66.879767,0.0,1322.0,0.00000,86,1.0,0.0,0.0,...,0.0,1.0,0.025784,0.05,0.025424,10.0,0.000000,0,10.0,3.571429
1,client_e547b89c05043229,content_08d7276f26f7df3c,23.778848,0.0,243.0,0.00000,57,0.0,0.0,0.0,...,0.0,1.0,0.038406,0.00,0.000000,10.0,778.613644,375,10.0,0.000000
2,client_e547b89c05043229,content_17d913cca79e48dc,38.052591,3.0,2654.0,0.00113,88,6.0,0.0,3.0,...,0.0,1.0,0.038406,0.00,0.000000,10.0,794.697136,375,10.0,0.000000


## 2. Fields: feature / label / context / excluded

To prevent data leakage and maintain a clean modeling pipeline, all dataset fields are sorted into four distinct categories:

### 1. Features (Model Inputs)
- **Content & Structure:** `word_count`, `content_depth`, `category_count`, `has_word_count`, `has_search_volume`.
- **Market & Keyword:** `search_volume`, `competition`, `cpc`, `keyword_opportunity`, `commercial_value`.
- **Freshness & Age:** `content_age_days`, `days_since_update`, `days_since_optimized`, `freshness_score`, `content_maturity`, `is_old`, `is_new`.
- **Relative & Peer-Group Metrics:** `cat_words_ratio`, `intent_backlinks_ratio`, `word_count_vs_bucket_median`, `search_volume_vs_bucket_mean`, etc.
- **Interactions:** `freshness_x_demand`, `depth_vs_competition`, `maturity_x_freshness`.

### 2. Label (Target Variable)
- `is_below_peer_median`: Binary target indicating whether the page's CTR is below the peer-group median for its position bucket.

### 3. Context (Metadata & IDs)
- `client_hash_id`: Used strictly for grouped cross-validation (`GroupShuffleSplit`) to prevent client data leakage.
- `content_hash_id`: Unique row identifier for tracking pages.
- `pos_bucket`: SERP position group used for calculating relative peer metrics.

### 4. Excluded Fields (With Reasons)
- `total_clicks`, `ctr`: **Reason:** Direct components of the outcome window/label; using them causes severe target leakage.
- `ga4_sessions`, `engaged_sessions`, `scroll_events`: **Reason:** Analytics metrics from the outcome window that represent future performance rather than pre-existing characteristics.
- `avg_position`: **Reason:** Used exclusively to derive structural `pos_bucket` and then dropped to avoid direct positioning leakage.

In [6]:
import pandas as pd
from pathlib import Path

# Load processed feature vector
processed_path = Path("../data/processed/refresh_feature_vector.csv")
df = pd.read_csv(processed_path)


# Define our categories for programmatic verification
features_sample = ["word_count", "search_volume", "freshness_score", "keyword_opportunity"]
context_cols = ["client_hash_id", "content_hash_id"]
excluded_suspects = ["total_clicks", "ctr", "ga4_sessions", "avg_position"]

print(" Field Bucket Verification:")
print(f"   - Total columns available in feature set: {df.shape[1]}")

# Check feature presence
missing_features = [f for f in features_sample if f not in df.columns]
print(f"   - Sample features check: {' All present' if not missing_features else f'Missing: {missing_features}'}")

# Check context columns presence
missing_context = [c for c in context_cols if c not in df.columns]
print(f"   - Context columns check: {' All present' if not missing_context else f'Missing: {missing_context}'}")

# Confirm excluded columns are safely dropped or separated
leaky_found = [e for e in excluded_suspects if e in df.columns and e in ["total_clicks", "ctr", "ga4_sessions"]]
if leaky_found:
    print(f"    Warning: Some outcome metrics are still in the dataframe columns: {leaky_found}")
else:
    print("    Safety Check Passed: Key leaky outcome fields are properly excluded from active features.")

 Field Bucket Verification:
   - Total columns available in feature set: 84
   - Sample features check:  All present
   - Context columns check:  All present


## 3. Verify it with queries (grain, counts, missing values, windows)

To ensure our data contracts are grounded in empirical evidence rather than assumptions, we execute programmatic verification queries across four key validation pillars:

1. **Grain Integrity Check:** 
   - *Claim:* Exactly one row represents one unique content entity.
   - *Query Verification:* Confirms that the total number of rows equals the unique count of `content_hash_id` with zero duplicates.

2. **Row Counts & Data Retention:** 
   - *Claim:* Filtering rules successfully eliminate statistical noise while preserving structural scale.
   - *Query Verification:* Evaluates dataset size before and after applying the core visibility filter (`total_impressions >= 100`)[cite: 1].

3. **Missing Value Audit:** 
   - *Claim:* Critical numeric and categorical fields are properly audited and handled to prevent downstream model execution errors.
   - *Query Verification:* Computes null/missing sums across core feature vectors.

4. **Temporal & Range Bounds Check:** 
   - *Claim:* Content age, update timelines, and traffic metrics fall within realistic physical and logical limits (no negative ages or impossible future updates).

In [11]:


print(f" Loaded source: {source_type} | Total records: {len(df):,}\n")

print(f"{'='*60}")
print("1. GRAIN & UNIQUENESS QUERY")
print(f"{'='*60}")
total_rows = len(df)
unique_contents = df["content_hash_id"].nunique() if "content_hash_id" in df.columns else 0
print(f"Total Rows: {total_rows:,}")
print(f"Unique content_hash_id: {unique_contents:,}")

if total_rows == unique_contents:
    print(" Grain Contract Passed: 1 row = 1 unique content item.")
else:
    print(" Grain Contract Failed: Duplicate content entities detected.")

print(f"\n{'='*60}")
print("2. MISSING VALUE AUDIT QUERY")
print(f"{'='*60}")
key_audit_cols = ["word_count", "search_volume", "content_age_days", "freshness_score", "cpc"]
existing_audit_cols = [c for c in key_audit_cols if c in df.columns]
missing_report = df[existing_audit_cols].isnull().sum()
print(missing_report.to_string())

print(f"\n{'='*60}")
print("3. TEMPORAL & RANGE BOUNDS QUERY")
print(f"{'='*60}")
if "content_age_days" in df.columns:
    print(f"Content Age (Days) -> Min: {df['content_age_days'].min()}, Max: {df['content_age_days'].max():,.0f}, Mean: {df['content_age_days'].mean():.1f}")
if "total_impressions" in df.columns:
    print(f"Total Impressions  -> Min: {df['total_impressions'].min():,.0f}, Max: {df['total_impressions'].max():,.0f}")

print("\n All structural queries executed successfully.")

 Loaded source: Processed Feature Vector | Total records: 118,092

1. GRAIN & UNIQUENESS QUERY
Total Rows: 118,092
Unique content_hash_id: 118,092
 Grain Contract Passed: 1 row = 1 unique content item.

2. MISSING VALUE AUDIT QUERY
word_count          0
search_volume       0
content_age_days    0
freshness_score     0
cpc                 0

3. TEMPORAL & RANGE BOUNDS QUERY
Content Age (Days) -> Min: 1, Max: 494, Mean: 194.1
Total Impressions  -> Min: 100, Max: 830,289

 All structural queries executed successfully.


## 4. Data limits

Every observational dataset has inherent boundaries. Acknowledging what the data *cannot* tell us is critical to maintaining honest scientific framing and preventing over-interpretation:

1. **Observational vs. Causal Blindness:** 
   - The data captures historical patterns and correlations (e.g., old, unoptimized content tending to have lower relative CTR), but it cannot prove direct causality. Recommending a content refresh is a directional decision-support mechanism, not a guaranteed traffic fix.

2. **GSC & Analytics Observability Gaps:** 
   - Relying on search engine logs means certain client subsets or pages have sparse auxiliary metrics. As accounted for via data completeness scoring (`data_completeness`) in our pipeline[cite: 1], rows lacking deep interaction signals require careful human interpretation.

3. **Selection Bias & Visibility Thresholds:** 
   - By enforcing a baseline visibility filter (`total_impressions >= 100`), brand-new content or ultra-low-traffic pages are intentionally excluded from the scoring queue[cite: 1]. Consequently, the model cannot evaluate or forecast performance for pages without established baseline history.

4. **Static Window Snapshots:** 
   - The 3-month observation window provides a cross-sectional view of performance, which cannot dynamically account for sudden external market shifts, unannounced core algorithm updates, or localized seasonality without external context.

In [12]:


print(f"{'='*60}")
print(f"4. DATA LIMITS & BOUNDARY AUDIT | Source: {source_info}")
print(f"{'='*60}")

# 2. Audit Observability Gap via data_completeness (engineered in 01_prepare_features.py)
if "data_completeness" in df.columns:
    completeness_stats = df["data_completeness"].describe()[['mean', 'min', '50%', 'max']]
    print("\n Observability Check (Data Completeness Ratio):")
    print(completeness_stats.to_string())
    print("   -> Limitation: Pages with lower completeness rely on imputed defaults.")
else:
    print("\n 'data_completeness' column not detected in current frame.")

# 3. Verify Visibility Filtering Bound (impressions >= 100 rule)
if "total_impressions" in df.columns:
    # Checking if raw data had lower values that were filtered out
    print(f"\n Total Active Records in Scored Vector: {len(df):,}")
    print("   -> Limitation Boundary: Enforces strict exclusion of zero/low-impression noise (< 100 impr.)[cite: 1].")

# 4. Audit Historical Age Span Boundaries
if "content_age_days" in df.columns:
    min_age = df['content_age_days'].min()
    max_age = df['content_age_days'].max()
    mean_age = df['content_age_days'].mean()
    print(f"\n Temporal Range Boundary (Content Age in Days):")
    print(f"   - Minimum Age: {min_age} days | Maximum Age: {max_age:,.0f} days | Mean: {mean_age:.1f} days")
    print("   -> Limitation: Extremely legacy content (> 5-10 years) may contain historical metadata anomalies.")

print("\n Data limits validation checks completed successfully.")

4. DATA LIMITS & BOUNDARY AUDIT | Source: Processed Feature Vector (01_prepare_features output)

 Observability Check (Data Completeness Ratio):
mean    0.492218
min     0.000000
50%     0.500000
max     1.000000
   -> Limitation: Pages with lower completeness rely on imputed defaults.

 Total Active Records in Scored Vector: 118,092
   -> Limitation Boundary: Enforces strict exclusion of zero/low-impression noise (< 100 impr.)[cite: 1].

 Temporal Range Boundary (Content Age in Days):
   - Minimum Age: 1 days | Maximum Age: 494 days | Mean: 194.1 days
   -> Limitation: Extremely legacy content (> 5-10 years) may contain historical metadata anomalies.

 Data limits validation checks completed successfully.
